<a href="https://colab.research.google.com/github/juli0AND/Evaluaci-n-comparativa-de-YOLOv5-y-YOLOv8/blob/main/Grad_CAM_YOLOV5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
%pip install -r requirements.txt
%pip install roboflow thop pandas -q
#IMPORTAR DATASET DE ROBOFLOW
from roboflow import Roboflow
rf = Roboflow(api_key="io4o0d2qhkzDHpqNE3Tb")
project = rf.workspace("deteccion-residuos").project("new-classification-yolov5_nano-pk95c")
version = project.version(1)
dataset = version.download("yolov5")
imagenes = (
    glob(f"{dataset.location}/test/images/*.jpg")
    + glob(f"{dataset.location}/test/images/*.jpeg")
    + glob(f"{dataset.location}/test/images/*.png")
)

print("Número de imágenes:", len(imagenes))
img_path = random.choice(imagenes)
print(img_path)
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_float = img.astype(np.float32) / 255.0

input_tensor = (
    torch.from_numpy(img_float)
    .permute(2,0,1)
    .unsqueeze(0)
    .float()
    .to(device)
)
class YOLOv5Wrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        pred = self.model.model(x)
        if isinstance(pred, tuple):
            pred = pred[0]
        return pred
		cam = EigenCAM(
    model=wrapped_model,
    target_layers=target_layers
)

grayscale_cam = cam(input_tensor=input_tensor)[0]

visualization = show_cam_on_image(
    img_float,
    grayscale_cam,
    use_rgb=True
)
import os
os.makedirs('gradcams', exist_ok=True)

imagenes_prueba = random.sample(imagenes, 20)

# Guardar la lista para usarla en YOLOv8
with open("imagenes_prueba.txt", "w") as f:
    for img in imagenes_prueba:
        f.write(img + "\n")

for i, img_path in enumerate(imagenes_prueba):

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_float = img.astype(np.float32) / 255.0

    input_tensor = (
        torch.from_numpy(img_float)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .float()
        .to(device)
    )

    grayscale_cam = cam(input_tensor=input_tensor)[0]

    visualization = show_cam_on_image(
        img_float,
        grayscale_cam,
        use_rgb=True
    )

    plt.figure(figsize=(8,8))
    plt.imshow(visualization)
    plt.axis('off')

    plt.savefig(
        f'gradcams/gradcam_{i+1}.png',
        dpi=1000,
        bbox_inches='tight',
        pad_inches=0
    )

    plt.close()
    print(f'Imagen {i+1} guardada')